# 04 — Part 2: BERT Semantic Feature Pipeline
**IS5126 LLM-Enhanced Credit Risk Assessment — NUS**

This notebook covers **Part 2, Pipelines A & B** on the ~200K desc subset:

| Pipeline | Description |
|---------|-------------|
| **A** | Traditional ML baseline on desc subset (no semantic features) |
| **B** | Naive (unfinetuned) BERT [CLS] embeddings + PCA → added as ML features |

**Output:** Table 1 — effect of BERT semantic features on AUC, KS, Gini, BACC, F1

**Design note:** We use unfinetuned BERT intentionally — this sets a lower bar so that the Qwen improvement in Notebook 05 is more meaningful and clearly attributable to model strength.

## 0. Setup

In [ ]:
!pip install -q transformers accelerate tqdm
!pip install -q xgboost lightgbm shap

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from tqdm.auto import tqdm

import torch
from transformers import BertTokenizer, BertModel

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, roc_curve, balanced_accuracy_score,
    f1_score, precision_score, recall_score
)
import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

COLORS = {
    'primary': '#1F4E79', 'accent': '#2E86AB',
    'good': '#28A745', 'bad': '#DC3545',
    'bert': '#9B59B6', 'neutral': '#6C757D',
    'xgb': '#2E86AB', 'lgb': '#28A745',
}

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR   = '/content/drive/MyDrive/is5126'
DATA_DIR    = f'{DRIVE_DIR}/data/processed'
MODEL_DIR   = f'{DRIVE_DIR}/models'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Config
BERT_MODEL_NAME   = 'bert-base-uncased'
BERT_MAX_LENGTH   = 128
BERT_BATCH_SIZE   = 64
BERT_PCA_N        = 50
RANDOM_STATE      = 42
TARGET_COL        = 'default'

# Cache paths
DESC_SUBSET_PATH  = f'{DATA_DIR}/desc_subset.parquet'
BERT_EMBED_PATH   = f'{DATA_DIR}/bert_embeddings.npy'
BERT_INDEX_PATH   = f'{DATA_DIR}/bert_embeddings_index.npy'
BERT_PCA_PATH     = f'{DATA_DIR}/bert_pca_features.parquet'

## 1. Extract Desc Subset (~200K loans)

In [ ]:
if os.path.exists(DESC_SUBSET_PATH):
    print("Loading cached desc subset...")
    df_desc = pd.read_parquet(DESC_SUBSET_PATH)
else:
    print("Loading raw LC data (this takes ~2-3 min)...")
    raw_path = f"{DRIVE_DIR}/accepted_2007_to_2018Q4.csv.gz"
    df_full = pd.read_csv(raw_path, low_memory=False)
    print(f"Raw data loaded: {len(df_full):,} rows")

    # Target variable
    status_map = {"Fully Paid": 0, "Charged Off": 1, "Default": 1}
    df_full["default"] = df_full["loan_status"].map(status_map)
    df_full = df_full[df_full["default"].notna()].copy()
    df_full["default"] = df_full["default"].astype(int)

    # Basic type cleaning
    df_full["int_rate"]   = pd.to_numeric(df_full["int_rate"].astype(str).str.replace("%",""), errors="coerce")
    df_full["revol_util"] = pd.to_numeric(df_full["revol_util"].astype(str).str.replace("%",""), errors="coerce")
    df_full["term"]       = df_full["term"].astype(str).str.extract(r"(\d+)").astype(float)

    # Derived columns expected by feature engineering
    df_full["emp_length_num"] = df_full["emp_length"].map({
        "< 1 year": 0.5, "1 year": 1, "2 years": 2, "3 years": 3, "4 years": 4,
        "5 years": 5, "6 years": 6, "7 years": 7, "8 years": 8, "9 years": 9, "10+ years": 10
    }).fillna(0)
    df_full["fico_score"] = (df_full["fico_range_low"] + df_full["fico_range_high"]) / 2
    issue_dt    = pd.to_datetime(df_full["issue_d"],        format="%b-%Y", errors="coerce")
    earliest_dt = pd.to_datetime(df_full["earliest_cr_line"], format="%b-%Y", errors="coerce")
    df_full["credit_history_months"] = ((issue_dt - earliest_dt).dt.days / 30).clip(lower=0)
    df_full["issue_date"] = issue_dt

    # Filter to rows with desc
    df_desc = df_full[df_full["desc"].notna() & (df_full["desc"].str.strip() != "")].copy()
    df_desc.reset_index(drop=True, inplace=True)
    df_desc.to_parquet(DESC_SUBSET_PATH, index=False)
    print(f"Saved desc subset: {len(df_desc):,} rows")

print(f"Desc subset: {df_desc.shape}")
print(f"Default rate: {df_desc[TARGET_COL].mean()*100:.2f}%")


In [ ]:
# Desc text stats
df_desc['desc_len'] = df_desc['desc'].str.split().str.len()
print('Desc word count stats:')
print(df_desc['desc_len'].describe())
print(f'\nWord count by default:')
print(df_desc.groupby(TARGET_COL)['desc_len'].mean().rename({0: 'Non-default', 1: 'Default'}))

## 2. Feature Engineering on Desc Subset

In [ ]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/is5126/src')
# Also try local if running from repo
sys.path.insert(0, '/content')

try:
    from features import create_derived_features
except ImportError:
    # Inline fallback
    def create_derived_features(df):
        df = df.copy()
        df['monthly_inc'] = df['annual_inc'] / 12
        df['installment_to_income'] = df['installment'] / (df['monthly_inc'] + 1)
        df['loan_to_income'] = df['loan_amnt'] / (df['annual_inc'] + 1)
        if 'tot_cur_bal' in df.columns:
            df['revol_to_total_bal'] = df['revol_bal'] / (df['tot_cur_bal'] + 1)
            df['avg_account_bal'] = df['tot_cur_bal'] / (df['total_acc'] + 1)
        df['delinq_per_year'] = df['delinq_2yrs'] / (df['credit_history_months'] / 12 + 1)
        df['inq_per_open_acc'] = df['inq_last_6mths'] / (df['open_acc'] + 1)
        df['revol_bal_to_income'] = df['revol_bal'] / (df['annual_inc'] + 1)
        if 'acc_open_past_24mths' in df.columns:
            df['new_acc_rate'] = df['acc_open_past_24mths'] / (df['total_acc'] + 1)
        df['is_long_term'] = (df['term'] == 60).astype(int)
        return df

df_desc = create_derived_features(df_desc)
print('Derived features created.')

In [ ]:
# Apply saved WoE maps (fit on Part 1 training set — reuse for consistency)
with open(f'{DATA_DIR}/woe_maps.json', 'r') as f:
    woe_maps = json.load(f)

WOE_CATEGORICALS = ['home_ownership', 'verification_status', 'purpose',
                    'initial_list_status', 'application_type', 'addr_state']

for feat in WOE_CATEGORICALS:
    if feat not in df_desc.columns:
        continue
    if feat not in woe_maps:
        continue
    woe_map = {k: float(v) for k, v in woe_maps[feat].items()}
    df_desc[f'{feat}_woe'] = df_desc[feat].astype(str).map(woe_map).fillna(0.0)

print('WoE encoding applied.')

In [ ]:
# Load feature list from Part 1 metadata
with open(f'{DATA_DIR}/feature_metadata.json', 'r') as f:
    feat_meta = json.load(f)

# Use no-grade features for comparability with Part 1 primary model
grade_related = feat_meta.get('grade_related', ['grade_num', 'sub_grade_num', 'int_rate'])
feature_cols = [c for c in feat_meta['feature_cols']
                if c not in grade_related and c in df_desc.columns]

print(f'Feature cols (no grade): {len(feature_cols)}')
missing = [c for c in feat_meta['feature_cols'] if c not in grade_related and c not in df_desc.columns]
if missing:
    print(f'Missing from desc subset (will skip): {missing}')

In [ ]:
# Train / test split
# Note: desc field only exists for 2008-2014 loans — time split yields no test data.
# We use stratified 80/20 random split instead, documented explicitly.
post_split = df_desc[df_desc['issue_date'] >= '2017-01-01'] if 'issue_date' in df_desc.columns else pd.DataFrame()
if len(post_split) / len(df_desc) < 0.05:
    print(f'Only {len(post_split)/len(df_desc)*100:.1f}% of desc data is post-2017.')
    print('Using stratified 80/20 random split (documented in report).')
    df_train, df_test = train_test_split(
        df_desc, test_size=0.2, random_state=RANDOM_STATE,
        stratify=df_desc[TARGET_COL]
    )
else:
    df_train = df_desc[df_desc['issue_date'] < '2017-01-01']
    df_test  = df_desc[df_desc['issue_date'] >= '2017-01-01']
    print('Using time-based split.')

df_train = df_train.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f'Train: {len(df_train):,} rows | default {df_train[TARGET_COL].mean()*100:.2f}%')
print(f'Test:  {len(df_test):,}  rows | default {df_test[TARGET_COL].mean()*100:.2f}%')

In [ ]:
X_train = df_train[feature_cols].fillna(0)
X_test  = df_test[feature_cols].fillna(0)
y_train = df_train[TARGET_COL]
y_test  = df_test[TARGET_COL]

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

## 3. Pipeline A — Traditional ML Baseline (No Semantics)

In [ ]:
def calc_ks(y_true, y_prob):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    return float(max(tpr - fpr))

def evaluate_full(y_true, y_prob, name='Model', threshold=None):
    """Full evaluation: AUC, KS, Gini, BACC, F1, Precision, Recall.
    threshold=None → use KS-optimal threshold.
    """
    auc = roc_auc_score(y_true, y_prob)
    ks  = calc_ks(y_true, y_prob)
    gini = 2 * auc - 1

    if threshold is None:
        fpr, tpr, thresholds = roc_curve(y_true, y_prob)
        threshold = thresholds[np.argmax(tpr - fpr)]

    y_pred = (y_prob >= threshold).astype(int)
    bacc  = balanced_accuracy_score(y_true, y_pred)
    f1    = f1_score(y_true, y_pred, zero_division=0)
    prec  = precision_score(y_true, y_pred, zero_division=0)
    rec   = recall_score(y_true, y_pred, zero_division=0)

    return {
        'Model': name,
        'AUC':       round(auc, 4),
        'KS':        round(ks, 4),
        'Gini':      round(gini, 4),
        'BACC':      round(bacc, 4),
        'F1':        round(f1, 4),
        'Precision': round(prec, 4),
        'Recall':    round(rec, 4),
        'Threshold': round(float(threshold), 4),
    }

In [ ]:
print('Training Pipeline A: XGBoost (no semantic features)...')

xgb_a = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE, n_jobs=-1,
    eval_metric='auc', early_stopping_rounds=50,
    verbosity=0
)
xgb_a.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

prob_a = xgb_a.predict_proba(X_test)[:, 1]
metrics_a = evaluate_full(y_test, prob_a, name='Pipeline A (XGB, no BERT)')
print(f'  AUC: {metrics_a["AUC"]} | KS: {metrics_a["KS"]} | Gini: {metrics_a["Gini"]} | BACC: {metrics_a["BACC"]}')

## 4. Pipeline B — BERT Semantic Feature Extraction

In [ ]:
def extract_bert_embeddings(texts, model, tokenizer, batch_size=64, max_length=128, device='cpu'):
    """Extract [CLS] token embeddings from BERT for a list of texts.
    Returns numpy array of shape (len(texts), 768).
    """
    model.eval()
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc='BERT embedding'):
        batch_texts = texts[i : i + batch_size]
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            outputs = model(**encoded)

        # [CLS] token = index 0 of last hidden state
        cls_emb = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_emb)

    return np.vstack(all_embeddings)

In [ ]:
# Check cache first — BERT on 200K takes ~30-60 min on Colab GPU
if os.path.exists(BERT_EMBED_PATH) and os.path.exists(BERT_INDEX_PATH):
    print('Loading cached BERT embeddings...')
    all_embeddings = np.load(BERT_EMBED_PATH)
    cached_ids = np.load(BERT_INDEX_PATH)
    print(f'Loaded: {all_embeddings.shape}')
else:
    print(f'Computing BERT embeddings on {len(df_desc):,} texts (device: {DEVICE})...')
    print('This will take ~30-60 min on Colab GPU. Grab a coffee.')

    tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_NAME)
    bert_model = BertModel.from_pretrained(BERT_MODEL_NAME).to(DEVICE)

    texts = df_desc['desc'].fillna('').tolist()
    all_embeddings = extract_bert_embeddings(
        texts, bert_model, tokenizer,
        batch_size=BERT_BATCH_SIZE, max_length=BERT_MAX_LENGTH, device=DEVICE
    )
    cached_ids = df_desc.index.to_numpy()

    np.save(BERT_EMBED_PATH, all_embeddings)
    np.save(BERT_INDEX_PATH, cached_ids)
    print(f'Saved embeddings: {all_embeddings.shape}')

    # Free GPU memory
    del bert_model
    torch.cuda.empty_cache()

In [ ]:
# Align embeddings with df_desc index
id_to_row = {idx: i for i, idx in enumerate(cached_ids)}
train_rows = [id_to_row[i] for i in df_train.index]
test_rows  = [id_to_row[i] for i in df_test.index]

bert_train_raw = all_embeddings[train_rows]
bert_test_raw  = all_embeddings[test_rows]
print(f'Train BERT shape: {bert_train_raw.shape}')
print(f'Test  BERT shape: {bert_test_raw.shape}')

In [ ]:
# PCA reduction: 768 → 50 dims
# Fit on train only to prevent leakage
print(f'PCA: 768 → {BERT_PCA_N} dims...')
pca = PCA(n_components=BERT_PCA_N, random_state=RANDOM_STATE)
bert_train_pca = pca.fit_transform(bert_train_raw)
bert_test_pca  = pca.transform(bert_test_raw)

explained = pca.explained_variance_ratio_.sum()
print(f'Variance explained by {BERT_PCA_N} components: {explained*100:.1f}%')

bert_cols = [f'bert_pc_{i}' for i in range(BERT_PCA_N)]
bert_train_df = pd.DataFrame(bert_train_pca, columns=bert_cols, index=df_train.index)
bert_test_df  = pd.DataFrame(bert_test_pca,  columns=bert_cols, index=df_test.index)

# Save PCA features for reuse in notebook 05
pd.concat([bert_train_df, bert_test_df]).to_parquet(BERT_PCA_PATH)
joblib.dump(pca, f'{MODEL_DIR}/bert_pca.joblib')
print('Saved PCA features and model.')

## 5. Pipeline B — ML with BERT Features

In [ ]:
# Augment feature matrix with BERT PCA components
X_train_b = pd.concat([X_train.reset_index(drop=True),
                        bert_train_df.reset_index(drop=True)], axis=1)
X_test_b  = pd.concat([X_test.reset_index(drop=True),
                        bert_test_df.reset_index(drop=True)], axis=1)

print(f'Feature matrix A: {X_train.shape[1]} cols')
print(f'Feature matrix B: {X_train_b.shape[1]} cols (+{BERT_PCA_N} BERT PCA)')

In [ ]:
print('Training Pipeline B: XGBoost + BERT features...')

xgb_b = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE, n_jobs=-1,
    eval_metric='auc', early_stopping_rounds=50,
    verbosity=0
)
xgb_b.fit(
    X_train_b, y_train,
    eval_set=[(X_test_b, y_test)],
    verbose=False
)

prob_b = xgb_b.predict_proba(X_test_b)[:, 1]
metrics_b = evaluate_full(y_test, prob_b, name='Pipeline B (XGB + BERT)')
print(f'  AUC: {metrics_b["AUC"]} | KS: {metrics_b["KS"]} | Gini: {metrics_b["Gini"]} | BACC: {metrics_b["BACC"]}')

In [ ]:
# SHAP importance for Pipeline B — which BERT components matter?
import shap
explainer_b = shap.TreeExplainer(xgb_b)
sample_idx = np.random.choice(len(X_test_b), min(3000, len(X_test_b)), replace=False)
shap_vals_b = explainer_b.shap_values(X_test_b.iloc[sample_idx])
if isinstance(shap_vals_b, list):
    shap_vals_b = shap_vals_b[1]

importance_b = pd.DataFrame({
    'feature': X_test_b.columns,
    'mean_abs_shap': np.abs(shap_vals_b).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)

bert_importance = importance_b[importance_b['feature'].str.startswith('bert_')]
print(f'Top 5 BERT components by SHAP importance:')
print(bert_importance.head(5).to_string(index=False))
print(f'\nBERT total SHAP share: {bert_importance["mean_abs_shap"].sum() / importance_b["mean_abs_shap"].sum() * 100:.1f}%')

## 6. Results — Table 1: Semantic vs No-Semantic

In [ ]:
results = [metrics_a, metrics_b]
results_df = pd.DataFrame(results).set_index('Model')

# Delta row
delta = {col: round(metrics_b[col] - metrics_a[col], 4)
         for col in ['AUC', 'KS', 'Gini', 'BACC', 'F1', 'Precision', 'Recall']
         if col in metrics_a}
delta['Model'] = 'Delta (B - A)'
delta['Threshold'] = '-'
results_df.loc['Delta (B - A)'] = pd.Series(delta).drop('Model')

print('=== TABLE 1: Effect of BERT Semantic Features ===')
print(results_df[['AUC', 'KS', 'Gini', 'BACC', 'F1', 'Precision', 'Recall']].to_string())

# Save
results_df.to_csv(f'{MODEL_DIR}/part2_table1_bert_results.csv')
print('\nSaved to Drive.')

In [ ]:
# ROC curve comparison
fig, ax = plt.subplots(figsize=(8, 6))

for prob, label, color in [
    (prob_a, 'Pipeline A: Traditional (AUC={:.4f})'.format(metrics_a['AUC']), COLORS['neutral']),
    (prob_b, 'Pipeline B: + BERT (AUC={:.4f})'.format(metrics_b['AUC']), COLORS['bert']),
]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    ax.plot(fpr, tpr, linewidth=2, label=label)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve: Pipeline A vs B (Desc Subset)', fontweight='bold')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{DRIVE_DIR}/figures/part2_roc_bert.png', dpi=150)
plt.show()
print('Figure saved.')

In [ ]:
# Statistical significance: DeLong test
from scipy import stats

def delong_roc_test(y_true, prob1, prob2):
    """Approximate DeLong test for AUC difference using bootstrap."""
    np.random.seed(RANDOM_STATE)
    n_bootstrap = 1000
    auc_diffs = []
    n = len(y_true)
    for _ in range(n_bootstrap):
        idx = np.random.choice(n, n, replace=True)
        try:
            a1 = roc_auc_score(y_true.iloc[idx], prob1[idx])
            a2 = roc_auc_score(y_true.iloc[idx], prob2[idx])
            auc_diffs.append(a1 - a2)
        except Exception:
            continue
    auc_diffs = np.array(auc_diffs)
    p_value = np.mean(auc_diffs <= 0)  # one-sided: B > A
    return p_value, np.std(auc_diffs)

p_val, std_diff = delong_roc_test(y_test, prob_b, prob_a)
auc_diff = metrics_b['AUC'] - metrics_a['AUC']
print(f'AUC difference (B - A): {auc_diff:+.4f}')
print(f'Bootstrap p-value (one-sided, B > A): {p_val:.4f}')
print(f'Significant at 0.05: {"YES" if p_val < 0.05 else "NO"}')